# 🛡️ PhishGuard — 20 Golden Features ML Training Pipeline (Kaggle Cloud)
### Program Kreativitas Mahasiswa (PKM) — MLOps Cloud Architecture

Notebook ini melatih model **RandomForestClassifier (150 Decision Trees)** dengan **20 Dimensi Fitur Emas Leksikal & Semantik Terpadu**:

| Kategori | Fitur yang Diekstraksi |
| :--- | :--- |
| **Struktural & Delimiter** | `url_length`, `hostname_length`, `path_length`, `count_dot`, `count_hyphen`, `count_at`, `count_question`, `count_equal`, `count_slash` |
| **Sintaksis & Reputasi** | `count_digits`, `count_subdomains`, `has_https`, `has_ip`, `has_port`, `has_url_shortener`, `is_suspicious_tld` |
| **Intelijen Leksikal & Teori Informasi** | `shannon_entropy` ⚡, `has_homograph` 🔤, `suspicious_words` 🚨, `indonesian_keywords` 🇮🇩 |

**Alur Pipeline:**
1. Setup dependensi & dataset (PhiUSIIL URL Corpus 235k dari UCI ML Repository)
2. Ekstraksi 20 fitur paralel multi-core berkecepatan tinggi
3. Stratified Data Split (70% Train, 15% Validation, 15% Test)
4. Komparasi Ilmiah: **Logistic Regression (Baseline)** vs **Random Forest 150 Trees (Proposed)**
5. Evaluasi performa lengkap & export gambar resolusi tinggi (300 DPI)
6. Real-World Live Sanity Check (Uji coba URL nyata Indonesia & global)
7. Ekspor berkas artefak `model_phishing.pkl` (~20 MB) & `metrics.json` siap unduh dari tab Output Kaggle.

In [ ]:
# [1] Setup Dependensi & Konfigurasi Lingkungan Kaggle
import os
import glob
import math
import json
import zipfile
import urllib.request
from collections import Counter
from urllib.parse import urlsplit

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("✓ Lingkungan Kaggle Siap. Scikit-Learn versi:", joblib.__version__)

In [ ]:
# [2] Akuisisi Dataset: Deteksi Input Kaggle atau Unduh Otomatis PhiUSIIL (UCI ML Repository)

dataset_df = None

# 1. Periksa apakah dataset sudah di-attach di /kaggle/input
kaggle_inputs = glob.glob("/kaggle/input/**/*.csv", recursive=True)
if kaggle_inputs:
    print(f"✓ Menemukan dataset di Kaggle Input: {kaggle_inputs[0]}")
    dataset_df = pd.read_csv(kaggle_inputs[0])
else:
    # 2. Unduh otomatis PhiUSIIL Phishing URL Dataset (235.795 URL) resmi dari UCI Repository
    print("ℹ Mengunduh PhiUSIIL URL Dataset resmi dari UCI ML Repository...")
    UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
    ZIP_PATH = "phiusiil_dataset.zip"
    EXTRACT_DIR = "dataset_extracted"
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    try:
        urllib.request.urlretrieve(UCI_ZIP_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)
        extracted_csv = glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True)
        if extracted_csv:
            dataset_df = pd.read_csv(extracted_csv[0])
            print(f"✓ Sukses memuat PhiUSIIL: {len(dataset_df):,} baris data!")
    except Exception as e:
        print(f"⚠️ Gagal mengunduh UCI zip ({e}), menggunakan dataset alternatif...")
        ALT_URL = "https://raw.githubusercontent.com/datasets/phishing-urls/main/data/dataset.csv"
        dataset_df = pd.read_csv(ALT_URL)

# Preprocessing Awal: Drop baris kosong dan duplikasi URL
url_col = "URL" if "URL" in dataset_df.columns else dataset_df.columns[0]
label_col = "label" if "label" in dataset_df.columns else dataset_df.columns[1]

dataset_df.dropna(subset=[url_col, label_col], inplace=True)
sebelum_dedup = len(dataset_df)
dataset_df.drop_duplicates(subset=[url_col], inplace=True)
print(f"✓ Pembersihan data: {sebelum_dedup - len(dataset_df):,} URL duplikat dihapus.")
print(f"✓ Total baris URL unik siap diolah: {len(dataset_df):,}")

In [ ]:
# [3] Definisi 20 Fitur Emas Leksikal, Sintaksis, dan Semantik

SUSPICIOUS_KEYWORDS = [
    "login", "verify", "account", "secure", "update", "signin",
    "banking", "password", "wallet", "confirm", "recover", "suspended",
    "webscr", "ebayisapi", "paypal", "credential"
]

INDO_KEYWORDS = [
    "undian", "bansos", "prakerja", "dana", "dana-kaget", "pulsa",
    "kejutan", "hadiah", "bca", "bri", "bni", "mandiri", "gopay",
    "shopee", "linkaja", "kuota", "subsidi", "blt", "ojk", "kominfo",
    "klikbca", "brimo"
]

URL_SHORTENERS = set([
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "ow.ly",
    "buff.ly", "rebrand.ly", "cutt.ly", "shorturl.at", "tiny.cc",
    "lnkd.in", "s.id", "rb.gy", "v.gd", "clck.ru", "rotf.lol"
])

SUSPICIOUS_TLDS = set([
    "xyz", "top", "tk", "ml", "ga", "cf", "gq", "work", "click",
    "buzz", "fit", "rest", "cam", "kim", "bar", "icu", "club",
    "live", "surf", "monster", "cfd"
])

CONFUSABLES = set("\u0430\u0441\u0435\u043e\u0440\u0455\u0445\u0456\u0458\u0443")

def calculate_shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    length = len(text)
    counts = Counter(text)
    return round(-sum((c / length) * math.log2(c / length) for c in counts.values()), 3)

def extract_20_features(url_str: str) -> dict:
    raw = str(url_str).strip()
    lowered = raw.lower()
    try:
        parsed = urlsplit(raw if "://" in raw else f"//{raw}")
        host = (parsed.hostname or "").rstrip(".").lower()
        port = parsed.port
        path = parsed.path or ""
    except:
        host, port, path = "", None, ""

    # Deteksi raw IPv4 numerik
    parts = host.split(".")
    is_ip = 1 if len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts) else 0

    # Subdomain depth
    subdomain_count = max(0, len([p for p in parts if p]) - 2)

    # Keyword hits
    susp_hits = sum(lowered.count(kw) for kw in SUSPICIOUS_KEYWORDS)
    indo_hits = sum(lowered.count(kw) for kw in INDO_KEYWORDS)

    # Homograph & Punycode
    has_homograph = 1 if ("xn--" in host or any(c in CONFUSABLES for c in host)) else 0

    # TLD risk & Shortener
    tld = parts[-1] if parts else ""
    is_suspicious_tld = 1 if tld in SUSPICIOUS_TLDS else 0
    is_shortener = 1 if any(host == s or host.endswith("." + s) for s in URL_SHORTENERS) else 0

    # Port check
    has_port = 1 if port is not None else 0

    # Shannon Entropy
    entropy = calculate_shannon_entropy(host)

    return {
        "url_length": len(raw),
        "hostname_length": len(host),
        "path_length": len(path),
        "count_dot": raw.count("."),
        "count_hyphen": raw.count("-"),
        "count_at": raw.count("@"),
        "count_question": raw.count("?"),
        "count_equal": raw.count("="),
        "count_slash": raw.count("/"),
        "count_digits": sum(char.isdigit() for char in raw),
        "count_subdomains": subdomain_count,
        "has_https": 1 if lowered.startswith("https://") else 0,
        "has_ip": is_ip,
        "has_port": has_port,
        "shannon_entropy": entropy,
        "has_homograph": has_homograph,
        "is_suspicious_tld": is_suspicious_tld,
        "has_url_shortener": is_shortener,
        "suspicious_words": susp_hits,
        "indonesian_keywords": indo_hits
    }

FEATURE_COLS = [
    "url_length", "hostname_length", "path_length", "count_dot", "count_hyphen",
    "count_at", "count_question", "count_equal", "count_slash", "count_digits",
    "count_subdomains", "has_https", "has_ip", "has_port", "shannon_entropy",
    "has_homograph", "is_suspicious_tld", "has_url_shortener", "suspicious_words",
    "indonesian_keywords"
]

print(f"✓ Konfigurasi 20 Fitur Emas Berhasil Diinisialisasi!")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:02d}. {f}")

In [ ]:
# [4] Ekstraksi Fitur Paralel Berkecepatan Tinggi (Kaggle Multi-Core CPU)
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

urls = dataset_df[url_col].astype(str).tolist()
raw_labels = dataset_df[label_col].tolist()

# Normalisasi Label: 1 = Phishing, 0 = Legitimate
# Pada dataset resmi PhiUSIIL UCI: Label 0 = Phishing, 1 = Legitimate
if "phiusiil" in str(kaggle_inputs).lower() or len(dataset_df) > 200000:
    print("ℹ Deteksi dataset PhiUSIIL UCI: memetakan Label 0 -> 1 (Phishing), Label 1 -> 0 (Legitimate).")
    y = np.array([1 if l == 0 else 0 for l in raw_labels])
else:
    y = np.array([1 if int(l) == 1 else 0 for l in raw_labels])

print(f"Mengekstrak 20 fitur dari {len(urls):,} URL dengan {os.cpu_count() or 4} thread CPU...")
with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
    extracted_list = list(tqdm(executor.map(extract_20_features, urls), total=len(urls)))

X = pd.DataFrame(extracted_list)[FEATURE_COLS]

# Stratified Data Splitting: 70% Train, 15% Validation, 15% Held-Out Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("=" * 50)
print("✓ Ekstraksi dan Pembagian Data Selesai!")
print(f"  - Train Set      : {len(X_train):,} baris ({np.mean(y_train)*100:.1f}% Phishing)")
print(f"  - Validation Set : {len(X_val):,} baris ({np.mean(y_val)*100:.1f}% Phishing)")
print(f"  - Test Set       : {len(X_test):,} baris ({np.mean(y_test)*100:.1f}% Phishing)")
print("=" * 50)

In [ ]:
# [5] Pelatihan Model: Logistic Regression (Baseline) vs RandomForestClassifier (150 Trees)

# 1. Model Baseline: Logistic Regression
print("[1/2] Melatih Model Baseline: Logistic Regression...")
baseline_model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"      Baseline Accuracy: {baseline_acc*100:.2f}% | F1-Score: {baseline_f1*100:.2f}%")

# 2. Model Utama yang Dipilih: RandomForestClassifier (150 Decision Trees)
print("[2/2] Melatih Model Utama: RandomForestClassifier (150 Trees Ensemble)...")
rf_model = RandomForestClassifier(
    n_estimators=150,
    criterion="gini",
    max_features="sqrt",
    min_samples_split=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train, y_train)
print("✓ Pelatihan RandomForestClassifier 150 Trees Selesai!")

In [ ]:
# [6] Evaluasi Ilmiah & Komparasi Model pada Data Uji (Test Set)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("=" * 60)
print("     TABEL HASIL KOMPARASI METODE UNTUK PROPOSAL PKM")
print("=" * 60)
print(f"1. Logistic Regression (Baseline) : Acc {baseline_acc*100:.2f}% | F1 {baseline_f1*100:.2f}%")
print(f"2. RandomForest (Proposed Model)  : Acc {acc*100:.2f}% | F1 {f1*100:.2f}% | Rec {rec*100:.2f}%")
print(f"   Peningkatan Performa F1        : +{(f1 - baseline_f1)*100:.2f}%")
print("=" * 60)

# Visualisasi 1: 20 Feature Importance Ranking
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(10, 7))
importances.plot(kind="barh", color="#1D4ED8")
plt.title("PhishGuard 20 Golden Feature Importances (Random Forest 150 Trees)", fontsize=13, fontweight="bold")
plt.xlabel("Gini Impurity Reduction Contribution")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.show()

# Visualisasi 2: Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Legitimate", "Phishing"], yticklabels=["Legitimate", "Phishing"])
plt.title("Confusion Matrix Test Set", fontweight="bold")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
# [7] Real-World Live Sanity Check: Uji Coba Langsung 5 URL Nyata

test_samples = [
    ("https://google.com/search?q=cybersecurity", "Google Search"),
    ("https://github.com/security", "GitHub Security"),
    ("http://192.168.1.100/login-update-paypal-banking-verify", "IP Host Attack"),
    ("http://dana-kaget-bansos-bri.xyz/login", "Indonesian Phishing Token"),
    ("http://xn--pple-43d.com/signin", "Homograph Cyrillic Spoof")
]

print("=" * 65)
print("        PENGUJIAN LANGSUNG URL NYATA PADA MODEL BARU")
print("=" * 65)
for sample_url, desc in test_samples:
    feats = extract_20_features(sample_url)
    feat_vector = pd.DataFrame([feats])[FEATURE_COLS]
    score = rf_model.predict_proba(feat_vector)[0][1]
    verdict = "PHISHING ⚠️" if score >= 0.50 else "LEGITIMATE 🛡️"
    print(f"[{desc}]")
    print(f"  URL   : {sample_url}")
    print(f"  Hasil : {verdict} (Skor Keyakinan: {score*100:.1f}%)")
    print("-" * 65)

In [ ]:
# [8] Ekspor Artefak Model (model_phishing.pkl & metrics.json) ke /kaggle/working/

# 1. Simpan Bundle model_phishing.pkl
output_bundle = {
    "model": rf_model,
    "feature_names": FEATURE_COLS,
    "threshold": 0.50,
    "algorithm": "RandomForestClassifier (150 Trees)",
    "metadata": {
        "dataset": "PhiUSIIL Phishing URL Dataset (UCI ML Repository)",
        "features_count": len(FEATURE_COLS),
        "n_estimators": 150,
        "training_environment": "Kaggle Cloud"
    }
}

joblib.dump(output_bundle, "model_phishing.pkl")
print(f"✓ Sukses mengekspor model_phishing.pkl ({os.path.getsize('model_phishing.pkl') / 1e6:.1f} MB)")

# 2. Simpan metrics.json
feature_rankings = [
    {"feature": feat, "importance": round(float(imp), 4), "rank": i + 1}
    for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).items())
]

metrics_dict = {
    "dataset": {
        "name": "PhiUSIIL Phishing URL Dataset (UCI ML Repository)",
        "total_samples": len(dataset_df),
        "features_count": len(FEATURE_COLS)
    },
    "model": {
        "algorithm": "RandomForestClassifier",
        "n_estimators": 150,
        "threshold": 0.50
    },
    "test_metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1_score": round(float(f1), 4),
        "roc_auc": round(float(roc), 4)
    },
    "confusion_matrix": {
        "true_negatives": int(cm[0][0]),
        "false_positives": int(cm[0][1]),
        "false_negatives": int(cm[1][0]),
        "true_positives": int(cm[1][1])
    },
    "feature_importance": feature_rankings
}

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2)
print("✓ Sukses mengekspor metrics.json")

print("=" * 60)
print("🎉 TRAINING CLOUD KAGGLE 20-FITUR SELESAI!")
print("Silakan unduh 'model_phishing.pkl' dan 'metrics.json' dari tab Output Kaggle")
print("lalu simpan ke folder artifacts/ di laptop Anda.")
print("=" * 60)